In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import PartialDependenceDisplay, partial_dependence
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
import os
warnings.filterwarnings('ignore')

# ============================================================================
# CREATE OUTPUT DIRECTORY
# ============================================================================
output_path = r"C:\Users\suhai\Desktop\PDP"
os.makedirs(output_path, exist_ok=True)

# ============================================================================
# LOAD DATA
# ============================================================================
print("="*70)
print("LOADING DATA")
print("="*70)

file_path = r"D:\2026 Work\Dr Hariharan Surendran\Paper 1 CS Glass Waste\Revision Work\Data\Data.csv"
df = pd.read_csv(file_path, encoding='ISO-8859-1')

print(f"Data Shape: {df.shape}")
print(f"\nColumn Names: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

# Identify target column
target_col = 'Compressive strength (Mpa)'
if target_col not in df.columns:
    similar = [c for c in df.columns if any(k in c.lower() for k in ['compressive', 'strength', 'mpa'])]
    target_col = similar[0] if similar else df.columns[-1]
    print(f"\nUsing '{target_col}' as target variable")

# ============================================================================
# REMOVE DEPENDENT VARIABLES - REMOVE DETERMINISTIC MATHEMATICAL DEPENDENCIES
# ============================================================================
print("\n" + "="*70)
print("REMOVING DETERMINISTIC MATHEMATICAL DEPENDENCIES")
print("="*70)

# Remove TGWP content and fly ash content as they are linearly dependent
# on replacement ratio and each other (mathematical artifacts of mixture design)
dependent_vars = ['TGWP content (kg/m3)', 'Fly ash (kg/m3)']
vars_to_remove = [col for col in dependent_vars if col in df.columns]

if vars_to_remove:
    df_clean = df.drop(columns=vars_to_remove)
    print(f"Removed variables with deterministic mathematical relationships: {vars_to_remove}")
    print("These variables are exact linear functions of TGWP replacement ratio and thus")
    print("create redundant information in the feature space.")
else:
    df_clean = df.copy()
    print("No deterministic dependency variables found to remove")

# Prepare features and target - using only non-redundant variables
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print(f"\nNon-redundant Features: {X.columns.tolist()}")
print(f"Target: {target_col}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# ============================================================================
# SPLIT DATA
# ============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# ============================================================================
# EXTRA TREES MODEL WITH SPECIFIED HYPERPARAMETERS
# ============================================================================
print("\n" + "="*70)
print("EXTRA TREES MODEL WITH HYPERPARAMETERS")
print("="*70)

# Define the model with your specified hyperparameters
model = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    bootstrap=False,
    max_features=0.7,
    ccp_alpha=0.001,
    random_state=42,
    n_jobs=-1
)

print("\nModel Parameters:")
print(model.get_params())

# ============================================================================
# TRAIN AND EVALUATE MODEL
# ============================================================================
print("\n" + "="*70)
print("TRAINING EXTRA TREES MODEL")
print("="*70)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate metrics
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

print(f"\nModel Performance:")
print(f"  Train R²:  {train_r2:.4f}")
print(f"  Test R²:   {test_r2:.4f}")
print(f"  Train MSE: {train_mse:.4f}")
print(f"  Test MSE:  {test_mse:.4f}")
print(f"  Train RMSE: {train_rmse:.4f}")
print(f"  Test RMSE:  {test_rmse:.4f}")
print(f"  Train MAE: {train_mae:.4f}")
print(f"  Test MAE:  {test_mae:.4f}")

# ============================================================================
# FEATURE IMPORTANCE
# ============================================================================
print("\n" + "="*70)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance.to_string(index=False))

# ============================================================================
# SET PLOT STYLE - IMPROVED FOR READABILITY
# ============================================================================
plt.style.use('default')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['figure.autolayout'] = True

# ============================================================================
# PLOT 1: FEATURE IMPORTANCE - IMPROVED
# ============================================================================
print("\nGenerating Feature Importance Plot...")

fig1, ax1 = plt.subplots(figsize=(14, 10))
n_top_features = len(feature_importance)
feature_imp_top = feature_importance.head(n_top_features)

# Create horizontal bar plot
bars = ax1.barh(range(len(feature_imp_top)), feature_imp_top['importance'].values, 
                color='steelblue', alpha=0.8, edgecolor='black', linewidth=0.8)
ax1.set_yticks(range(len(feature_imp_top)))
ax1.set_yticklabels(feature_imp_top['feature'], fontsize=11)
ax1.set_xlabel('Feature Importance', fontsize=13, fontweight='bold', labelpad=10)
ax1.set_title(f'Feature Importance - Extra Trees Model\nTest R²: {test_r2:.4f}', 
             fontsize=14, fontweight='bold', pad=15)
ax1.grid(True, alpha=0.3, axis='x')
ax1.invert_yaxis()

# Add value labels on bars
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax1.text(width + 0.002, bar.get_y() + bar.get_height()/2, 
             f'{width:.4f}', va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(output_path, '01_Feature_Importance.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# ============================================================================
# PLOT 2: ACTUAL VS PREDICTED - IMPROVED
# ============================================================================
print("\nGenerating Actual vs Predicted Plot...")

fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(16, 7))

# Train set
ax2a.scatter(y_train, y_pred_train, alpha=0.6, color='steelblue', edgecolors='black', linewidth=0.3, s=50)
ax2a.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', linewidth=2.5)
ax2a.set_xlabel('Actual Values', fontsize=13, fontweight='bold', labelpad=10)
ax2a.set_ylabel('Predicted Values', fontsize=13, fontweight='bold', labelpad=10)
ax2a.set_title(f'Training Set\nR² = {train_r2:.4f}, RMSE = {train_rmse:.4f}', fontsize=13, fontweight='bold', pad=12)
ax2a.grid(True, alpha=0.3)
ax2a.tick_params(axis='both', labelsize=10)

# Test set
ax2b.scatter(y_test, y_pred_test, alpha=0.6, color='coral', edgecolors='black', linewidth=0.3, s=50)
ax2b.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2.5)
ax2b.set_xlabel('Actual Values', fontsize=13, fontweight='bold', labelpad=10)
ax2b.set_ylabel('Predicted Values', fontsize=13, fontweight='bold', labelpad=10)
ax2b.set_title(f'Test Set\nR² = {test_r2:.4f}, RMSE = {test_rmse:.4f}', fontsize=13, fontweight='bold', pad=12)
ax2b.grid(True, alpha=0.3)
ax2b.tick_params(axis='both', labelsize=10)

plt.suptitle('Extra Trees Model: Actual vs Predicted Values', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(output_path, '02_Actual_vs_Predicted.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# ============================================================================
# PLOT 3: RESIDUAL PLOTS - IMPROVED
# ============================================================================
print("\nGenerating Residual Plots...")

fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(16, 7))

# Training residuals
residuals_train = y_train - y_pred_train
ax3a.scatter(y_pred_train, residuals_train, alpha=0.6, color='steelblue', edgecolors='black', linewidth=0.3, s=50)
ax3a.axhline(y=0, color='red', linestyle='--', linewidth=2.5)
ax3a.set_xlabel('Predicted Values', fontsize=13, fontweight='bold', labelpad=10)
ax3a.set_ylabel('Residuals', fontsize=13, fontweight='bold', labelpad=10)
ax3a.set_title('Training Set Residuals', fontsize=13, fontweight='bold', pad=12)
ax3a.grid(True, alpha=0.3)
ax3a.tick_params(axis='both', labelsize=10)

# Test residuals
residuals_test = y_test - y_pred_test
ax3b.scatter(y_pred_test, residuals_test, alpha=0.6, color='coral', edgecolors='black', linewidth=0.3, s=50)
ax3b.axhline(y=0, color='red', linestyle='--', linewidth=2.5)
ax3b.set_xlabel('Predicted Values', fontsize=13, fontweight='bold', labelpad=10)
ax3b.set_ylabel('Residuals', fontsize=13, fontweight='bold', labelpad=10)
ax3b.set_title('Test Set Residuals', fontsize=13, fontweight='bold', pad=12)
ax3b.grid(True, alpha=0.3)
ax3b.tick_params(axis='both', labelsize=10)

plt.suptitle('Extra Trees Model: Residual Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(output_path, '03_Residual_Plots.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# ============================================================================
# 2D PDP PLOTS - SELECTED NON-REDUNDANT VARIABLE PAIRS
# ============================================================================
print("\n" + "="*70)
print("GENERATING 2D JOINT PARTIAL DEPENDENCE PLOTS")
print("="*70)
print("\nNOTE: These are selected two-variable combinations that avoid")
print("deterministic mathematical dependencies from the mixture formulation.")
print("Statistical correlations between variables may still exist in the dataset.")

# First, let's see what features we have
print("\nAvailable features in dataset:")
for i, col in enumerate(X.columns):
    print(f"  {i+1}. '{col}'")

# Define the specific feature pairs we want - avoiding deterministic dependencies
# These are non-redundant pairs that don't have exact mathematical relationships
feature_pairs = [
    ('Replacement (%)', 'Age (Days)'),  # TGWP replacement ratio × curing age
    ('Replacement (%)', 'Na2SiO3/NaOH Molarity (m)'),  # TGWP replacement ratio × NaOH concentration
    ('Replacement (%)', 'W/b ratio (Kg/m3)'),  # TGWP replacement ratio × water-to-binder ratio
    ('Age (Days)', 'Na2SiO3/NaOH Molarity (m)'),  # curing age × NaOH concentration
    ('Na2SiO3/NaOH Molarity (m)', 'W/b ratio (Kg/m3)')  # NaOH concentration × water-to-binder ratio
]

# Map display names for publication-quality labels
# IMPORTANT: 'Replacement (%)' is displayed as 'TGWP replacement ratio (%)' in plots
display_names = {
    'Replacement (%)': 'TGWP replacement ratio (%)',
    'Age (Days)': 'Curing age (days)',
    'Na2SiO3/NaOH Molarity (m)': 'NaOH concentration (M)',
    'W/b ratio (Kg/m3)': 'Water-to-binder ratio'
}

# Verify which pairs exist
valid_pairs = []
for feature1, feature2 in feature_pairs:
    if feature1 in X.columns and feature2 in X.columns:
        valid_pairs.append((feature1, feature2))
        print(f"✓ Valid pair: {feature1} × {feature2}")
    else:
        if feature1 not in X.columns:
            print(f"✗ Feature '{feature1}' not found in dataset")
        if feature2 not in X.columns:
            print(f"✗ Feature '{feature2}' not found in dataset")

print(f"\nGenerating {len(valid_pairs)} two-dimensional PDP plots")
print("These plots show the joint partial dependence of the model's predictions")
print("on selected non-redundant variable pairs.")

if len(valid_pairs) > 0:
    # Create figure for 2D PDPs
    n_plots = len(valid_pairs)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig_pdp, axes_pdp = plt.subplots(
        nrows=n_rows,
        ncols=n_cols,
        figsize=(18, 5.5 * n_rows),
        constrained_layout=True
    )
    
    # Flatten axes for easier iteration
    if n_plots > 1:
        axes_flat = axes_pdp.flatten()
    else:
        axes_flat = [axes_pdp]
    
    # Generate each 2D PDP plot
    for idx, (feature1, feature2) in enumerate(valid_pairs):
        try:
            feature1_idx = list(X.columns).index(feature1)
            feature2_idx = list(X.columns).index(feature2)
            
            # Get display labels
            label1 = display_names.get(feature1, feature1)
            label2 = display_names.get(feature2, feature2)
            
            print(f"\nGenerating: {label1} × {label2}")
            
            # Create 2D PDP with appropriate grid resolution
            # Adjust grid based on unique values to avoid interpolation errors
            n_unique1 = X[feature1].nunique()
            n_unique2 = X[feature2].nunique()
            grid_res = min(20, max(5, min(n_unique1, n_unique2)))
            
            # Create 2D PDP - using only the main title, no subtitle
            disp = PartialDependenceDisplay.from_estimator(
                estimator=model,
                X=X_train,
                features=[(feature1_idx, feature2_idx)],
                feature_names=X.columns,
                grid_resolution=grid_res,
                ax=axes_flat[idx]
            )
            
            # Customize plot - clean, no subtitle
            axes_flat[idx].set_title(f'{label1} × {label2}', 
                                    fontsize=12, fontweight='bold', pad=10)
            axes_flat[idx].set_xlabel(label1, fontsize=11, fontweight='bold', labelpad=8)
            axes_flat[idx].set_ylabel(label2, fontsize=11, fontweight='bold', labelpad=8)
            axes_flat[idx].tick_params(axis='both', labelsize=10)
            
            # Add colorbar label
            if len(axes_flat[idx].collections) > 0:
                cbar = axes_flat[idx].collections[0].colorbar
                if cbar is not None:
                    cbar.set_label(f'Effect on {target_col}', fontsize=11, fontweight='bold', labelpad=8)
                    cbar.ax.tick_params(labelsize=10)
            
            print(f"  ✓ Successfully generated")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            axes_flat[idx].set_visible(False)
    
    # Hide unused subplots
    for j in range(idx + 1, len(axes_flat)):
        if j < len(axes_flat):
            fig_pdp.delaxes(axes_flat[j])
    
    # Add main title only - clean and publication-ready
    plt.suptitle('Two-Dimensional Partial Dependence Plots for the Extra Trees Model', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, '06_PDP_2D_Interaction.png'), 
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print("\n✅ 2D PDP plots generated successfully")
    print("\nIMPORTANT NOTE: These plots show joint partial dependence, not formal")
    print("feature interactions. To quantify true interactions, Friedman's H-statistic")
    print("or similar interaction measures should be computed separately.")
else:
    print("\n❌ No valid pairs found. Please check the feature names.")
    print("Available features:", X.columns.tolist())

# ============================================================================
# PLOT 4: RESIDUAL DISTRIBUTION - IMPROVED
# ============================================================================
print("\nGenerating Residual Distribution Plot...")

fig4, ax4 = plt.subplots(figsize=(12, 7))

# Histogram of residuals
n_bins = min(30, int(np.sqrt(len(residuals_test))))
ax4.hist(residuals_test, bins=n_bins, alpha=0.7, color='coral', edgecolor='black', linewidth=0.8, density=True)
ax4.axvline(x=0, color='red', linestyle='--', linewidth=2.5, label='Zero Error')
ax4.set_xlabel('Residuals', fontsize=13, fontweight='bold', labelpad=10)
ax4.set_ylabel('Density', fontsize=13, fontweight='bold', labelpad=10)
ax4.set_title('Test Set Residual Distribution', fontsize=14, fontweight='bold', pad=15)
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=11)
ax4.tick_params(axis='both', labelsize=10)

# Add statistics
mean_residual = np.mean(residuals_test)
std_residual = np.std(residuals_test)
skewness = pd.Series(residuals_test).skew()
kurtosis = pd.Series(residuals_test).kurtosis()

stats_text = f'Mean: {mean_residual:.4f}\nStd: {std_residual:.4f}\nSkewness: {skewness:.4f}\nKurtosis: {kurtosis:.4f}'
ax4.text(0.95, 0.95, stats_text, transform=ax4.transAxes, 
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'),
         fontsize=10, fontfamily='monospace')

plt.tight_layout()
plt.savefig(os.path.join(output_path, '04_Residual_Distribution.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# ============================================================================
# PLOT 5: SINGLE FEATURE PDP PLOTS
# ============================================================================
print("\n" + "="*70)
print("GENERATING SINGLE FEATURE PDP PLOTS")
print("="*70)

n_features = len(X.columns)
n_cols = min(2, n_features)
n_rows = (n_features + n_cols - 1) // n_cols

fig1, axes1 = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(14, 5 * n_rows),
    constrained_layout=True
)

if n_features > 1:
    axes1 = axes1.flatten()
else:
    axes1 = [axes1]

for i, feature in enumerate(X.columns):
    try:
        feature_idx = list(X.columns).index(feature)
        
        # Create PDP plot
        disp = PartialDependenceDisplay.from_estimator(
            estimator=model,
            X=X_train,
            features=[feature_idx],
            feature_names=X.columns,
            grid_resolution=50,
            line_kw={"color": "red", "linewidth": 3},
            pd_line_kw={"color": "red", "linewidth": 3, "label": "PDP"},
            ax=axes1[i]
        )
        
        current_ax = axes1[i]
        
        # Get display label
        display_label = display_names.get(feature, feature)
        
        # Customize plot
        current_ax.set_title(f'PDP for {display_label}\n(Importance: {feature_importance.iloc[i]["importance"]:.4f})', 
                           fontsize=11, fontweight='bold', pad=12)
        current_ax.set_xlabel(display_label, fontsize=11, fontweight='bold', labelpad=8)
        current_ax.set_ylabel(f'Effect on {target_col}', fontsize=11, fontweight='bold', labelpad=8)
        current_ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
        current_ax.legend(loc='best', fontsize=9)
        current_ax.tick_params(axis='both', labelsize=9)
        
        # Add actual data distribution as rug plot
        feature_values = X_train.iloc[:, feature_idx]
        y_min, y_max = current_ax.get_ylim()
        rug_height = (y_max - y_min) * 0.03
        current_ax.scatter(feature_values, 
                         np.full_like(feature_values, y_min + rug_height), 
                         alpha=0.15, s=8, color='blue', marker='|',
                         label='Data Distribution')
        
    except Exception as e:
        print(f"Error plotting {feature}: {e}")
        axes1[i].set_visible(False)

# Remove empty subplots
for j in range(i + 1, len(axes1)):
    if j < len(axes1):
        fig1.delaxes(axes1[j])

plt.suptitle(f'Single Feature PDP Plots - Extra Trees Model', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(output_path, '05_PDP_Single_Feature.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# ============================================================================
# SAVE RESULTS TO CSV
# ============================================================================
print("\n" + "="*70)
print("SAVING RESULTS TO CSV")
print("="*70)

# Save feature importance
feature_importance.to_csv(os.path.join(output_path, 'feature_importance.csv'), index=False)

# Save model performance
performance_df = pd.DataFrame({
    'Metric': ['R²', 'MSE', 'RMSE', 'MAE'],
    'Train': [train_r2, train_mse, train_rmse, train_mae],
    'Test': [test_r2, test_mse, test_rmse, test_mae]
})
performance_df.to_csv(os.path.join(output_path, 'model_performance.csv'), index=False)

# Save predictions
predictions_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred_test,
    'Residuals': residuals_test
})
predictions_df.to_csv(os.path.join(output_path, 'test_predictions.csv'), index=False)

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*70)
print("EXTRA TREES PDP ANALYSIS COMPLETE")
print("="*70)

print("\n✅ All files saved to:", output_path)
print("\n📊 Generated Files:")
print("="*60)
print("\n📈 Plots:")
print("  01_Feature_Importance.png - Feature importance bar plot")
print("  02_Actual_vs_Predicted.png - Actual vs predicted scatter plots")
print("  03_Residual_Plots.png - Residual analysis plots")
print("  04_Residual_Distribution.png - Residual distribution histogram")
print("  05_PDP_Single_Feature.png - Single feature PDP plots")
if len(valid_pairs) > 0:
    print(f"  06_PDP_2D_Interaction.png - Two-dimensional PDP plots ({len(valid_pairs)} selected non-redundant pairs)")

print("\n📊 CSV Files:")
print("  feature_importance.csv - Feature importance values")
print("  model_performance.csv - Model performance metrics")
print("  test_predictions.csv - Test set predictions and residuals")

print("\n📈 Model Performance:")
print(f"  Test R²:  {test_r2:.4f}")
print(f"  Test RMSE: {test_rmse:.4f}")
print(f"  Test MAE:  {test_mae:.4f}")

print("\n" + "="*70)
print("✅ COMPLETE! All files saved to your desktop folder: PDP")
print("="*70)

LOADING DATA
Data Shape: (192, 9)

Column Names: ['Fly ash (Kg/m3)', 'Toughened glass waste powder (Kg/m3)', 'Replacement (%)', 'M-sand', 'W/b ratio (Kg/m3)', 'water (Kg/m3)', 'Na2SiO3/NaOH Molarity (m)', 'Age (Days)', 'Compressive strength (Mpa)']

First 5 rows:
   Fly ash (Kg/m3)  Toughened glass waste powder (Kg/m3)  Replacement (%)  \
0              500                                     0                0   
1              400                                   100               20   
2              300                                   200               40   
3              200                                   300               60   
4              100                                   400               80   

   M-sand  W/b ratio (Kg/m3)  water (Kg/m3)  Na2SiO3/NaOH Molarity (m)  \
0    1375               0.36            180                          8   
1    1375               0.36            180                          8   
2    1375               0.36            180        